In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
import torch.nn as nn
import pandas as pd
torch.manual_seed(42)

In [ ]:
device =torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
df =pd.read_csv('/content/sample_data/mnist_train_small.csv')
df.head()

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.shape

(19999, 785)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(df.drop('6',axis=1),df['6'],test_size=0.2,random_state=42)
x_train=x_train/255.0
x_test=x_test/255.0


In [ ]:
# class customdataset(Dataset):
#   def __init__(self,features,labels):
#     self.features =torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
#     self.labels =torch.tensor(labels,dtype=torch.long)
#   def __len__(self):
#     return len(self.features)
#   def __getitem__(self,idx):
#     return self.features[idx],self.labels[idx]


class customdataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(
            features.values,  # or features.to_numpy()
            dtype=torch.float32
        ).reshape(-1, 1, 28, 28)

        self.labels = torch.tensor(
            labels.values if hasattr(labels, "values") else labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


In [ ]:
train_dataset =customdataset(x_train,y_train)
test_dataset =customdataset(x_test,y_test)

In [ ]:
train_loader =DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader =DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [ ]:
class MYCNN(nn.Module):
    def __init__(self, num_features=1):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(num_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
learning_rate =0.01
epochs =25

In [ ]:
model =MYCNN(1)
model.to(device)

criterion =nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [ ]:
#training loop
for i in range(epochs):
  total_epochs_loss =0
  for batch_features,batch_labels in train_loader:
    batch_features =batch_features.to(device)
    batch_labels =batch_labels.to(device)


    output =model(batch_features)
    loss =criterion(output,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_epochs_loss+=loss.item()
  avg_loss =total_epochs_loss/len(train_loader)
  print(f"Epoch {i+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/25, Loss: 0.3603
Epoch 2/25, Loss: 0.1269
Epoch 3/25, Loss: 0.0890
Epoch 4/25, Loss: 0.0639
Epoch 5/25, Loss: 0.0521
Epoch 6/25, Loss: 0.0424
Epoch 7/25, Loss: 0.0358
Epoch 8/25, Loss: 0.0318
Epoch 9/25, Loss: 0.0265
Epoch 10/25, Loss: 0.0220
Epoch 11/25, Loss: 0.0187
Epoch 12/25, Loss: 0.0171
Epoch 13/25, Loss: 0.0156
Epoch 14/25, Loss: 0.0144
Epoch 15/25, Loss: 0.0129
Epoch 16/25, Loss: 0.0106
Epoch 17/25, Loss: 0.0096
Epoch 18/25, Loss: 0.0100
Epoch 19/25, Loss: 0.0080
Epoch 20/25, Loss: 0.0071
Epoch 21/25, Loss: 0.0072
Epoch 22/25, Loss: 0.0075
Epoch 23/25, Loss: 0.0062
Epoch 24/25, Loss: 0.0060
Epoch 25/25, Loss: 0.0066


In [ ]:
model.eval()  # Set model to evaluation mode

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

avg_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.0421
Test Accuracy: 98.80%
